# Exercise 6: Code Interpreter Tool -- Data Analysis in a Sandbox

The `code_interpreter` tool lets the model write and execute Python code in a  
sandboxed container. Here we pass CSV data inline and ask the model to perform  
analysis -- calculating trends, ratios, and identifying at-risk accounts.

In [1]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

## Send CSV data with analysis instructions

In [2]:
response = client.responses.create(
    model="gpt-4.1",
    tools=[{
        "type": "code_interpreter",
        "container": {"type": "auto"},
    }],
    instructions=(
        "Use the code_interpreter tool for all calculations. "
        "Use print() to display results clearly. "
        "After the code runs, present a concise final summary."
    ),
    input="""Analyze this CSV data using code_interpreter:

Customer,Tickets_Q1,Tickets_Q2,Tickets_Q3,Tickets_Q4,Contract_Value,Churn_Risk
Acme Corp,12,8,15,22,500000,High
Globex Inc,3,2,4,3,1200000,Low
Initech,7,12,18,25,300000,Critical
Wayne Enterprises,1,1,2,1,2000000,Low
Umbrella Corp,5,8,6,9,800000,Medium
Stark Industries,2,3,2,4,1500000,Low
Cyberdyne,20,25,30,35,400000,Critical

Calculate: total tickets, trend (increasing/decreasing/stable), tickets per $100K contract value.
Identify the top 3 accounts needing immediate attention. Print a summary table.
""",
)

## Inspect the output flow

The response interleaves `code_interpreter_call` items (the code that was executed)  
and `message` items (the model's narrative summary).

In [3]:
print("=== Output flow ===\n")
all_text_parts = []
for i, item in enumerate(response.output):
    if item.type == "message":
        for content in item.content:
            if content.type == "output_text":
                all_text_parts.append(content.text)
                label = content.text[:100].replace("\n", " ")
                print(f"[{i}] MESSAGE: {label}...")
    elif item.type == "code_interpreter_call":
        print(f"[{i}] CODE_INTERPRETER (status={item.status})")
        print(f"    Container: {item.container_id}")
        code_lines = item.code.strip().split("\n")
        print(f"    Code: {len(code_lines)} lines")
        # Show first and last few lines
        for line in code_lines[:3]:
            print(f"      {line}")
        if len(code_lines) > 6:
            print(f"      ... ({len(code_lines) - 6} more lines)")
        for line in code_lines[-3:]:
            print(f"      {line}")
    print()

=== Output flow ===

[0] CODE_INTERPRETER (status=completed)
    Container: cntr_69a665d7e9e88193b735630ea13599700a2c2d7c7a9f01a2
    Code: 62 lines
      import pandas as pd
      import numpy as np
      
      ... (56 more lines)
      print(top3_attention[['Customer', 'Total_Tickets', 'Trend', 'Tickets_per_100K', 'Churn_Risk']].to_string(index=False))
      
      summary_df, top3_attention[['Customer', 'Total_Tickets', 'Trend', 'Tickets_per_100K', 'Churn_Risk']]

[1] MESSAGE: It appears the ticket columns were loaded as object types (not numeric), which caused an issue with ...

[2] CODE_INTERPRETER (status=completed)
    Container: cntr_69a665d7e9e88193b735630ea13599700a2c2d7c7a9f01a2
    Code: 34 lines
      # Convert ticket columns to numeric
      for col in ticket_columns:
          df[col] = pd.to_numeric(df[col])
      ... (28 more lines)
      print(top3_attention[['Customer', 'Total_Tickets', 'Trend', 'Tickets_per_100K', 'Churn_Risk']].to_string(index=False))
      
     

## Full analysis output

In [4]:
print("=" * 60)
print("FULL ANALYSIS")
print("=" * 60)
for part in all_text_parts:
    print(part)
    print()

FULL ANALYSIS
It appears the ticket columns were loaded as object types (not numeric), which caused an issue with the trend calculation. I'll convert these columns to numeric and re-run the analysis.



## Token usage and output item count

In [ ]:
print(f"Tokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out")
print(f"Output items: {len(response.output)}")

## Interview one-liner

> **`code_interpreter`** gives the model a sandboxed Python runtime so it can write, execute, and iterate on code autonomously -- perfect for data analysis, math validation, and any task where you want verifiable computation rather than hallucinated numbers.